# 00 — Environment Setup

**Goal:** Set up a professional Python development environment for NLP and data science work.

In this notebook you will:
1. Set up Python virtual environments (venv + conda)
2. Install core packages (numpy, pandas, scikit-learn, spacy, nltk)
3. Configure Jupyter and Git
4. Verify everything works with a simple test

This is the foundation — get this right and every following notebook will just work.

Every notebook in this series runs on the stack you assemble here: a Python interpreter >= 3.10, an isolated environment, the numeric/ML/NLP libraries, spaCy's English model, NLTK's data packs, and Git for versioning. The cells below verify each layer in turn, so when something fails in a later chapter you know exactly which layer to blame.

**Why a solid environment matters:** nothing you build later — tokenizers, parsers, the ATS scoring engine — runs without this foundation, and a reproducible setup is what lets you ship consistent output. It is also a standard screening topic: interviewers probe how you manage virtual environments, package versions, and model/data downloads. Get this chapter right and every following notebook "just works" — that is the point.

## 1. Checking Your Python Installation

Python 3.10+ is the floor for this curriculum: the type hints, dataclasses, and modern typing used throughout (plus current NumPy, pandas, and spaCy releases) all assume it. The cell prints the interpreter's identity so a version mismatch never becomes a mystery later.

**What the code does:**
- `sys.version` prints the full build string — the stored output shows **Python 3.12.3** on **win32**.
- `sys.executable` resolves the exact interpreter path (`c:\Users\MSI\AppData\Local\Programs\Python\Python312\python.exe` here) — the first thing to check when `python` on your PATH points somewhere unexpected.
- `assert sys.version_info >= (3, 10)` hard-stops the notebook on older interpreters with a clear message.

**Try it:** the trailing `✓ Python version OK` line confirms the assertion passed on this machine. Run the same cell in any other environment you plan to use for the project.

In [1]:
import sys
print(f"Python version: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Platform: {sys.platform}")

# Check we're on 3.10+
assert sys.version_info >= (3, 10), "Need Python 3.10+"
print("✓ Python version OK")

Python version: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
Executable: c:\Users\MSI\AppData\Local\Programs\Python\Python312\python.exe
Platform: win32
✓ Python version OK


## 2. Virtual Environments

Virtual environments isolate project dependencies. Two approaches:

### Option A: venv (lightweight, built-in)
```bash
python -m venv .venv
# Activate:
#   Windows: .venv\Scripts\activate
#   macOS/Linux: source .venv/bin/activate
```

### Option B: Conda (heavier, better for data science)
```bash
conda create -n resanalyze python=3.11
conda activate resanalyze
```

We'll use the current environment — but knowing how to create them is essential.

**Why isolate at all:** different projects pin different — sometimes conflicting — versions of NumPy or spaCy. An environment keeps each project's dependencies self-contained, and a `requirements.txt` or `environment.yml` makes the setup reproducible for anyone cloning the repo.

| | venv | conda |
|---|---|---|
| Ships with | Python stdlib | Anaconda / Miniconda |
| Package source | pip (PyPI) | conda + pip |
| Best for | lightweight, single-version projects | data/ML stacks with non-Python deps |

The exact tool matters less than consistency: this repository's notebooks assume one activated environment, with every dependency either already installed or installed by the next cell.

## 3. Install Core Packages

This list is the project's dependency contract: `numpy`/`pandas` for tabular data, `scikit-learn` for ML models, `spacy`/`nltk` for NLP, `matplotlib`/`seaborn` for visualization, and `jupyter`/`nbformat` for the notebook tooling itself.

**What the code does:**
- `importlib.import_module(pkg.replace("-", "_"))` probes each package without importing at module level — an `ImportError` means *not installed*.
- On failure it falls back to `subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])`, so the notebook self-heals on a fresh machine.
- The stored output shows most packages already present; `scikit-learn` was installed on the fly.

**Why it matters:** every later notebook opens with imports from this list — if a cell here fails, fix it before moving on, not in Ch. 04.

In [2]:
import subprocess, sys, importlib

packages = [
    "numpy", "pandas", "scikit-learn",
    "spacy", "nltk", "matplotlib", "seaborn",
    "jupyter", "nbformat"
]

for pkg in packages:
    try:
        importlib.import_module(pkg.replace("-", "_"))
        print(f"  ✓ {pkg} already installed")
    except ImportError:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"  ✓ {pkg} installed")

print("All core packages ready!")

  ✓ numpy already installed
  ✓ pandas already installed
  Installing scikit-learn...
  ✓ scikit-learn installed
  ✓ spacy already installed
  ✓ nltk already installed
  ✓ matplotlib already installed
  ✓ seaborn already installed
  ✓ jupyter already installed
  ✓ nbformat already installed
All core packages ready!


## 4. Download spaCy Language Model

spaCy ships as code *plus* a trained model: `en_core_web_sm` is the small English pipeline whose components (tagger, parser, NER) power Ch. 10–12. The `sm` suffix means speed over accuracy; `md`/`lg` trade up as needed.

**What the code does:**
- `spacy.load("en_core_web_sm")` inside `try`/`except OSError` — a missing model raises `OSError`, which triggers a one-time `python -m spacy download`, then loads.
- A smoke test parses a sentence: the stored output shows 8 tokens and the entity `('NLP', 'ORG')` — spaCy's NER already recognizes *NLP* as an organization, zero extra code.

**Try it:** entity extraction like this is exactly the Ch. 12 named-entity material — running here with one line.

In [3]:
# Download the small English model for spaCy
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    print("✓ spaCy model 'en_core_web_sm' already loaded")
except OSError:
    print("Downloading spaCy model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    import spacy
    nlp = spacy.load("en_core_web_sm")
    print("✓ spaCy model downloaded and loaded")

# Quick test
doc = nlp("This is a test sentence for NLP.")
print(f"Tokens: {[t.text for t in doc]}")
print(f"Entities: {[(e.text, e.label_) for e in doc.ents]}")

✓ spaCy model 'en_core_web_sm' already loaded
Tokens: ['This', 'is', 'a', 'test', 'sentence', 'for', 'NLP', '.']
Entities: [('NLP', 'ORG')]


## 5. NLTK Data Download

NLTK is code plus *data*: tokenizers, stopword lists, and WordNet are downloaded once per machine. Each `nltk.download(...)` call is idempotent, and `quiet=True` suppresses repeat-download chatter.

**What the code does:**
- Pulls `punkt_tab` (sentence/word tokenizer data), `stopwords`, `wordnet` (the Ch. 08 lemmatizer's lexicon), and `averaged_perceptron_tagger_eng` (the Ch. 10 POS tagger).
- The final import verifies the download: the stored output reports **198 English stopwords** — the exact list Ch. 07 will filter on.

**Try it:** rerun the cell — downloads are skipped once the data exists, which is why reruns stay quiet.

In [4]:
import nltk
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk.corpus import stopwords
print(f"✓ NLTK ready — {len(stopwords.words('english'))} English stopwords available")

✓ NLTK ready — 198 English stopwords available


## 6. Git Configuration

Git records *who* changed what: `user.name` and `user.email` are stamped onto every commit. Unset, commits get a generic identity and `git blame` / PR tooling becomes useless.

**What the code does:**
- Two `git config --global` calls set the identity; `--global` applies to every repo on this machine.
- `git config --list | findstr user` filters the config to the identity lines — the Windows cousin of `grep user`.
- The stored output shows the placeholder values: **replace `Your Name` / `your.email@example.com` with your real identity** before your first commit.

**Try it:** run `git config user.name` afterwards to confirm the value stuck.

In [ ]:
# Replace the values below with your own name and email
!git config --global user.name "Your Name"
!git config --global user.email "your.email@example.com"
!git config --list | findstr user
print("\u2713 Git configured")

user.name=Your Name
user.email=your.email@example.com
\u2713 Git configured


## 7. Project Structure Overview

This is your working directory. Every notebook lives in a numbered folder:

```
notebooks/
├── part1_foundations/
│   ├── block_a/    (00-03: Environment & Tooling)
│   ├── block_b/    (04-12: NLP Pipeline)
│   ├── block_c/    (13-20: Text Representation)
│   └── block_d/    (21-25: Semantic Embeddings)
└── part2_intelligence/
    ├── block_e/    (26-31: Document Parsing)
    ├── block_f/    (32-39: Resume Information Extraction)
    ├── block_g/    (40-45: Job Description Intelligence)
    ├── block_h/    (46-49: Semantic Matching)
    └── block_i/    (50-54: ATS Scoring Engine)
```

The layout mirrors the learning path: **Part I** builds the NLP toolbox (foundations → pipeline → text representation → embeddings), **Part II** assembles it into the ATS stack (parsing → extraction → matching → scoring). Each numbered folder is one notebook; `block_a` (00–03) is the environment and language tooling this chapter is standing on.

Keep this map in mind when later chapters reference "the extractor from Ch. 03" or "the scoring engine from Ch. 54" — every piece has a home, and the numbering keeps dependencies one-directional.

## 8. Quick Sanity Check

The final gate: import every core library in one cell and print its version. This turns "my code broke" into "which library is the wrong version" — the first question in any debugging session.

**What the code does:**
- One import block covering the whole stack, then `__version__` per library: the stored output shows NumPy **1.26.4**, pandas **2.2.3**, scikit-learn **1.7.2**, spaCy **3.8.14**, NLTK **3.10.0**.
- Those version numbers pin what the rest of the curriculum was written and tested against.

**Try it:** run this after any environment change (new machine, new conda env, `pip upgrade`) — it should always end with the ready message.

In [6]:
import numpy as np
import pandas as pd
import sklearn
import spacy
import nltk

print(f"NumPy:      {np.__version__}")
print(f"Pandas:     {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"spaCy:      {spacy.__version__}")
print(f"NLTK:       {nltk.__version__}")

print("✓ Environment is ready. Let's go! 🚀")

NumPy:      1.26.4
Pandas:     2.2.3
scikit-learn: 1.7.2
spaCy:      3.8.14
NLTK:       3.10.0
✓ Environment is ready. Let's go! 🚀


## Key Insight

**A reproducible environment is the silent dependency of every result in this curriculum.**

Everything that follows — tokenizers, parsers, the scoring engine — runs on exactly this stack: interpreter, isolated environment, pinned library versions, downloaded models, and a committed Git identity. When a later notebook misbehaves, this chapter is the checklist: version mismatch, missing model, unset identity. With the foundation verified, the next chapter sharpens the language the pipeline is written in — Ch. 01, Python Engineering Refresher.